In [2]:
import numpy as np
from scipy import constants
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
from tweezer_functions import * 
from IonChainTools import *
from scipy.optimize import fsolve
import matplotlib.colors as mcolors
import matplotlib.colorbar as mcolorbar
from scipy.optimize import fsolve
from scipy.optimize import curve_fit
from scipy.optimize import minimize
import matplotlib.ticker as ticker
import itertools

#Constants in SI units
eps0 = constants.epsilon_0 
m = 39.9626*constants.atomic_mass
c = constants.c
e = constants.e
hbar = constants.hbar
pi = np.pi

# setting up parameters that we're not changing
qubit_wavelength = 729e-9
tweezer_wavelength = 532e-9
omega_tweezer = 2*pi*c/tweezer_wavelength
print(omega_tweezer)
df = pd.read_csv("S_P_only.csv",sep = ",",encoding = "UTF-8")
lambdares = np.array(df["wavelength (nm)"])*1e-9
omega_res = 2*pi*c/lambdares
linewidths = np.array(df["A_ki (s^-1)"])
lifetimes = linewidths
print(linewidths)
#test

3540698434791077.0
[1.47e+08 1.40e+08]


# Now I want to sweep over N ions, limit the maximum number of tweezer beams to 2, and fix one cooling ion

I know the cooling time per mode is inversely proportional to the max of lamb-dicke parameter of mode (max meaning I'm picking out which ion in each mode has the maximum lamb-dicke)

I'm assuming that the k-vector part of the lamb-dicke parameter is equal per ion, so I'm only going to consider the mode-vector coupling term for now

This means I'll pick out the mode-vector coupling term per mode to find the time per mode

Then I'll sum up all of those times per mode to get the total time

In [ ]:
def tweezer_combos_full_radial(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N_list,
    f_rf_r,
    P_opt,
    w0,
    max_tweezed=1,
    qubit_wavelength=729e-9,
    theta=0
):
    """
    Compute all tweezer combinations and corresponding radial mode frequencies,
    including power dependence and tweezed/untweezed mode separation.

    mode_calc_r is expected to return a list of tuples:
        [(freq1, eigvec1), (freq2, eigvec2), ...]
    """

    import numpy as np
    import itertools
    import pandas as pd

    # --- Normalize inputs ---
    if np.isscalar(N_list):
        N_list = [int(N_list)]
    if np.isscalar(P_opt):
        P_opt = [P_opt]

    pi = np.pi
    rows = []

    # --- Helper functions ---
    def max_eig_index(eigvec):
        """Return max amplitude and its ion index"""
        idx = np.argmax(np.abs(eigvec))
        return eigvec[idx], idx

    def max_eig_index_no_tweeze(eigvec, tweezed):
        N = len(eigvec)
        ion_indices = np.arange(N)
        mask = ~np.isin(ion_indices, tweezed)  # untweezed ions only
        if np.any(mask):
            eig_untweezed = eigvec[mask]
            ions_untweezed = ion_indices[mask]
            idx_local = np.argmax(np.abs(eig_untweezed))
            return eig_untweezed[idx_local], ions_untweezed[idx_local]
        else:
            return np.nan, np.nan

    def sum_no_tweeze(arr, tweezed, N):
        ion_indices = np.arange(N)
        mask = ~np.isin(ion_indices, tweezed)
        if np.any(mask):
            return np.sqrt(np.sum(arr[mask] ** 2))
        else:
            return np.nan

    # --- Loop over number of ions ---
    for N in N_list:
        # Generate all possible tweezer combinations
        all_combos = []
        for r in range(0, max_tweezed + 1):
            all_combos.extend(itertools.combinations(range(N), r))

        # RF trap setup
        w_rf_r = f_rf_r * 2 * pi
        w_rf_r_list = np.full(N, w_rf_r)
        ueq = ion_spacing(N, f_rf_r)[0]

        # --- Loop over optical powers ---
        for P_total in P_opt:
            for tweezed_positions in all_combos:
                n_tweezed = len(tweezed_positions)
                P_per = P_total / n_tweezed if n_tweezed > 0 else 0.0

                # Compute tweezer potential for this configuration
                pot = potential(omega_tweezer, linewidths, omega_res, P_per, w0)
                w_tw_r = omega_tweezer_r(pot, w0, m)

                # Combine tweezed and untweezed radial frequencies
                combo = np.array([
                    np.sqrt(w_tw_r**2 + w_rf_r_list[i]**2) if i in tweezed_positions else w_rf_r_list[i]
                    for i in range(N)
                ])

                # --- Compute radial modes ---
                modes = mode_calc_r(m, combo, ueq, N)

                # Extract frequencies and eigenvectors
                freqs = np.array([f for f, v in modes], dtype=float)
                eigvecs = np.vstack([np.ravel(v) for f, v in modes])  # shape (n_modes, N)

                # --- Initialize row ---
                row = {
                    "N": N,
                    #"P_total (W)": P_total,
                    "Tweezed ions": tweezed_positions,
                    "P_per_tweezer (W)": P_per,
                    #"Combined radial frequencies": combo,
                }

                # --- Loop over modes dynamically ---
                for mode_index, (f, eigvec) in enumerate(modes):
                    eigvec = np.ravel(eigvec)
                    mode_sum_no_tweeze = sum_no_tweeze(eigvec, tweezed_positions, N)
                    mode_max, mode_max_index = max_eig_index(eigvec)
                    mode_max_no_tweeze, mode_max_no_tweeze_index = max_eig_index_no_tweeze(eigvec, tweezed_positions)

                    # Store per-mode data
                    row[f"Mode{mode_index}_freq"] = f
                    row[f"Mode{mode_index}_eigvec"] = eigvec
                    #row[f"Mode{mode_index}_max"] = mode_max
                    #row[f"Mode{mode_index}_max_index"] = mode_max_index
                    #row[f"Mode{mode_index}_max_no_tweeze"] = mode_max_no_tweeze
                    #row[f"Mode{mode_index}_max_no_tweeze_index"] = mode_max_no_tweeze_index

                # --- Unique mode→ion mapping (by absolute value, preserving sign) ---
                abs_eigs = np.abs(eigvecs)
                used = set()
                mode_to_ion_indices = []
                mode_to_ion_amplitudes = []

                for mode_i in range(abs_eigs.shape[0]):
                    order = np.argsort(abs_eigs[mode_i])[::-1]
                    for idx in order:
                        if idx not in used:
                            # pick by largestx |amplitude| but store signed value
                            mode_to_ion_indices.append(int(idx))
                            mode_to_ion_amplitudes.append(float(eigvecs[mode_i, idx]))
                            used.add(idx)
                            break

                row["Mode→Ion indices"] = mode_to_ion_indices
                row["Mode→Ion amplitudes"] = mode_to_ion_amplitudes

                # --- Maximum (by |amplitude|) of assigned amplitudes ---
                row["Max amplitude (mode→ion)"] = (
                    max(map(abs, mode_to_ion_amplitudes)) if mode_to_ion_amplitudes else np.nan
                )

                # --- Store completed row ---
                rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
N = np.arange(3,4)
P_opt = [1]
data= tweezer_combos_full_radial(omega_tweezer,linewidths,omega_res,m,mode_calc_r,N,f_rf_r,P_opt,w0)

In [ ]:
data

In [ ]:
def compute_mode_coupling_row(row, qubit_wavelength=729e-9, theta_deg=0):
    amps = np.atleast_1d(row.get('Mode→Ion amplitudes', []))
    # gather matching Mode{i}_freq entries (0 -> Mode0_freq, 1 -> Mode1_freq, ...)
    freqs = []
    for i in range(len(amps)):
        freqs.append(row.get(f"Mode{i}_freq", np.nan))
    amps = np.array(amps, dtype=float)
    freqs = np.array(freqs, dtype=float)

    # if any freq missing return array of nans with same length as amps
    if amps.size == 0 or np.any(np.isnan(freqs)):
        return np.full(amps.shape, np.nan, dtype=float)

    # prefactor and corrected sqrt (sqrt(hbar / (2*m*omega)))
    prefactor = (2 * np.pi * qubit_wavelength / c) * np.cos(np.radians(theta_deg))
    coupling = amps * prefactor * np.sqrt(hbar / (2 * m * freqs))
    return coupling

# create column with per-row arrays
data['Mode→Ion_coupling'] = data.apply(_compute_mode_coupling_row, axis=1)


In [ ]:
compute_mode_coupling_row()
data['Mode→Ion_coupling'] = data.apply(compute_mode_coupling_row, axis=1)


In [ ]:
#next step: take the list of amplitudes and convert it to times, plot the minimum time per N and sweep over N

# look at this one later, it doubles up on indices for a tweezed ion

In [ ]:
# ...existing code...
def tweezer_combos_full_radial_with_mode_mapping_ignore_tweezed(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N_list,
    f_rf_r,
    P_opt,
    w0,
    max_tweezed=1,
    qubit_wavelength=729e-9,
    theta=0,
):
    """
    Computes radial modes and eigenvectors and tracks:
    - Max per mode and no-tweeze max
    - Mode→Ion mapping ignoring tweezed ions (duplicates allowed)
    - Mode→Ion amplitudes and max amplitude
    - Mode→Ion couplings (per-row array and per-mode Mode{i}_coupling columns)
    """

    import numpy as np
    import itertools
    import pandas as pd
    from scipy import constants as _const

    c = _const.c
    hbar = _const.hbar

    if np.isscalar(N_list):
        N_list = [int(N_list)]
    if np.isscalar(P_opt):
        P_opt = [P_opt]

    pi = np.pi
    rows = []

    def max_eig_index(eigvec):
        idx = np.argmax(np.abs(eigvec))
        return eigvec[idx], idx

    def max_eig_index_no_tweeze(eigvec, tweezed):
        N = len(eigvec)
        ion_indices = np.arange(N)
        mask = ~np.isin(ion_indices, tweezed)
        if np.any(mask):
            eig_untweezed = eigvec[mask]
            ions_untweezed = ion_indices[mask]
            idx_local = np.argmax(np.abs(eig_untweezed))
            return eig_untweezed[idx_local], ions_untweezed[idx_local]
        else:
            return np.nan, np.nan

    for N in N_list:
        all_combos = []
        for r in range(0, max_tweezed + 1):
            all_combos.extend(itertools.combinations(range(N), r))

        w_rf_r = f_rf_r * 2 * pi
        w_rf_r_list = np.full(N, w_rf_r)
        ueq = ion_spacing(N, f_rf_r)[0]

        for P_total in P_opt:
            for tweezed_positions in all_combos:
                n_tweezed = len(tweezed_positions)
                P_per = P_total / n_tweezed if n_tweezed > 0 else 0.0

                pot = potential(omega_tweezer, linewidths, omega_res, P_per, w0)
                w_tw_r = omega_tweezer_r(pot, w0, m)

                combo = np.array([
                    np.sqrt(w_tw_r**2 + w_rf_r_list[i]**2) if i in tweezed_positions else w_rf_r_list[i]
                    for i in range(N)
                ])

                modes = mode_calc_r(m, combo, ueq, N)

                # frequencies and eigenvector matrix (n_modes, N)
                freqs = np.array([f for f, v in modes], dtype=float)
                eigvecs = np.vstack([np.ravel(v) for f, v in modes]) if len(modes) else np.empty((0, N))

                row = {
                    "N": N,
                    "P_total (W)": P_total,
                    "Tweezed ions": tweezed_positions,
                    "P_per_tweezer (W)": P_per,
                    "Combined radial frequencies": combo,
                }

                mode_to_ion_indices = []
                mode_to_ion_amplitudes = []

                for mode_index, (f, eigvec) in enumerate(modes):
                    eigvec = np.ravel(eigvec)
                    mode_max, mode_max_index = max_eig_index(eigvec)
                    mode_max_no_tweeze, mode_max_no_tweeze_index = max_eig_index_no_tweeze(eigvec, tweezed_positions)

                    row[f"Mode{mode_index}_max"] = mode_max
                    row[f"Mode{mode_index}_max_index"] = mode_max_index
                    row[f"Mode{mode_index}_max_no_tweeze"] = mode_max_no_tweeze
                    row[f"Mode{mode_index}_max_no_tweeze_index"] = mode_max_no_tweeze_index

                    # --- Mode→Ion mapping ignoring tweezed ions ---
                    candidates = [i for i in range(N) if i not in tweezed_positions]
                    if not candidates:
                        mode_to_ion_indices.append(np.nan)
                        mode_to_ion_amplitudes.append(np.nan)
                    else:
                        max_idx_local = candidates[np.argmax(np.abs(eigvec[candidates]))]
                        mode_to_ion_indices.append(int(max_idx_local))
                        mode_to_ion_amplitudes.append(float(eigvec[max_idx_local]))

                row["Mode→Ion indices"] = mode_to_ion_indices
                row["Mode→Ion amplitudes"] = mode_to_ion_amplitudes
                row["Max amplitude (mode→ion)"] = max(
                    map(abs, [x for x in mode_to_ion_amplitudes if not np.isnan(x)])
                ) if any(not np.isnan(x) for x in mode_to_ion_amplitudes) else np.nan

                # --- Compute mode→ion coupling (per-row array + per-mode columns) ---
                k = 2 * np.pi / qubit_wavelength
                proj = np.cos(np.radians(theta))
                couplings = []
                for i, amp in enumerate(mode_to_ion_amplitudes):
                    # amplitude may be nan
                    try:
                        amp_f = float(amp)
                    except Exception:
                        amp_f = np.nan

                    freq_i = freqs[i] if (i < len(freqs)) else np.nan

                    if np.isnan(amp_f) or np.isnan(freq_i) or freq_i == 0:
                        val = np.nan
                    else:
                        # use sqrt(hbar/(2*m*omega)) where freqs are angular frequencies expected from mode_calc_r
                        val = amp_f * k * proj * np.sqrt(hbar / (2 * m * freq_i))

                    couplings.append(float(val) if not np.isnan(val) else np.nan)
                    row[f"Mode{i}_coupling"] = float(val) if not np.isnan(val) else np.nan

                row["Mode→Ion_coupling"] = couplings

                rows.append(row)

    return pd.DataFrame(rows)
# ...existing code...


In [ ]:
N = np.arange(3,4)
P_opt = [100e-3]
data= tweezer_combos_full_radial_with_mode_mapping_ignore_tweezed(omega_tweezer,linewidths,omega_res,m,mode_calc_r,N,f_rf_r,P_opt,w0)

In [ ]:
data

In [ ]:
#what is the two photon rabi frequency for each of these?????
#effective rabi frequency, g = eta*Omega/2 from Sideband Thermometry of Ion Crystals -- PRX Quantum published in 2023
#and regular Rabi frequency is= 
# Rabi_squared = [(linewidths[0]*6*pi*c**2)/(hbar*omega_res[0]**3) * I , linewidths[1] *(6*pi*c**2)/(hbar*omega_res[1]**3) * I]
#but i need the linewidths and omegares for the qubit transition 
#or refer to equation 2.17 from the Hempel thesis
#also, use I = 2P/(pi w0^2)

1. find the saturation intensity of the qubit transition
2. insert the linewidth and resonant frequency of the qubit transition
3. find the rabi frequency with the equation I used above
4. take my mode vector contribution values and convert them to lamb-dicke factors
5. find the effective rabi frequency

linewidth_729 = 136e-3*2*pi
omega_res_729 = 

# brute force calculation of every single ion coupling to every single mode -- radial only

In [ ]:

def tweezer_combos_full_radial(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N_list,
    f_rf_r,
    P_opt,
    w0,
    max_tweezed=1,
):
    """
    Compute all tweezer combinations and corresponding radial mode frequencies,
    including power dependence and tweezed/untweezed mode separation.

    mode_calc_r is expected to return a list of tuples:
        [(freq1, eigvec1), (freq2, eigvec2), ...]
    This version ensures the dataframe has Mode0_eigvec, Mode1_eigvec, ... Mode{N-1}_eigvec
    and Mode0_freq, Mode1_freq, ... Mode{N-1}_freq for each row (missing entries filled with NaN).
    """


    # --- Normalize inputs ---
    if np.isscalar(N_list):
        N_list = [int(N_list)]
    if np.isscalar(P_opt):
        P_opt = [P_opt]

    pi = np.pi
    rows = []

    # --- Loop over number of ions ---
    for N in N_list:
        # Generate all possible tweezer combinations
        all_combos = []
        for r in range(0, max_tweezed + 1):
            all_combos.extend(itertools.combinations(range(N), r))

        # RF trap setup
        w_rf_r = f_rf_r * 2 * pi
        w_rf_r_list = np.full(N, w_rf_r)
        ueq = ion_spacing(N, f_rf_r)[0]

        # --- Loop over optical powers ---
        for P_total in P_opt:
            for tweezed_positions in all_combos:
                n_tweezed = len(tweezed_positions)
                P_per = P_total / n_tweezed if n_tweezed > 0 else 0.0

                # Compute tweezer potential for this configuration
                pot = potential(omega_tweezer, linewidths, omega_res, P_per, w0)
                w_tw_r = omega_tweezer_r(pot, w0, m)

                # Combine tweezed and untweezed radial frequencies
                combo = np.array([
                    np.sqrt(w_tw_r**2 + w_rf_r_list[i]**2) if i in tweezed_positions else w_rf_r_list[i]
                    for i in range(N)
                ])

                # --- Compute radial modes ---
                modes = mode_calc_r(m, combo, ueq, N)

                # Extract frequencies and eigenvectors
                freqs = np.array([f for f, v in modes], dtype=float) if len(modes) else np.array([], dtype=float)
                if len(modes):
                    eigvecs = np.vstack([np.ravel(v) for f, v in modes])  # shape (n_modes, N)
                else:
                    eigvecs = np.empty((0, N))

                # --- Initialize row with shared info ---
                row = {
                    "N": N,
                    "Tweezed ions": tweezed_positions,
                    "P_per_tweezer (W)": P_per,
                    "Combined radial frequencies": combo,
                }

                # --- Ensure columns for all possible modes up to N exist per row ---
                # Fill Mode{i}_freq and Mode{i}_eigvec for i in [0, N-1]
                for mode_index in range(N):
                    # frequency
                    if mode_index < len(freqs):
                        row[f"Mode{mode_index}_freq"] = float(freqs[mode_index])
                    else:
                        row[f"Mode{mode_index}_freq"] = np.nan

                    # eigenvector (length N) or NaN array
                    if mode_index < eigvecs.shape[0]:
                        row[f"Mode{mode_index}_eigvec"] = np.ravel(eigvecs[mode_index]).astype(float)
                    else:
                        # use full-length nan array to keep shape consistent
                        row[f"Mode{mode_index}_eigvec"] = np.full(N, np.nan, dtype=float)

                # --- Store completed row ---
                rows.append(row)

    return pd.DataFrame(rows)
# ...existing code...
def build_mode_series_and_combinations(df, max_modes=None):
    """
    Extract per-mode lists of (df_index, eigvec_array) from df.

    Returns a dict with:
      - mode_series: mapping mode_index -> original pandas Series (unchanged)
      - mode_lists:  mapping mode_index -> list of tuples (df_index, np.ndarray(eigvec))
    If max_modes is set, only modes with index < max_modes are returned.
    """


    # find Mode{i}_eigvec columns sorted by i
    mode_cols = sorted(
        [c for c in df.columns if re.match(r"^Mode\d+_eigvec$", c)],
        key=lambda c: int(re.match(r"Mode(\d+)_eigvec$", c).group(1)),
    )
    mode_indices = [int(re.match(r"Mode(\d+)_eigvec$", c).group(1)) for c in mode_cols]

    if max_modes is not None:
        mode_indices = [i for i in mode_indices if i < int(max_modes)]

    # keep the original Series for convenience
    mode_series = {i: df[f"Mode{i}_eigvec"] for i in mode_indices}

    # build lists of (original_index, np.array(value)) for each mode
    mode_lists = {}
    for i in mode_indices:
        col = f"Mode{i}_eigvec"
        items = []
        if col in df.columns:
            for idx in df.index:
                val = df.at[idx, col]
                # convert to a numeric numpy array (works if stored as list/ndarray/scalar)
                try:
                    arr = np.asarray(val, dtype=float)
                except Exception:
                    # fall back to object array if conversion fails
                    arr = np.atleast_1d(val)
                items.append((idx, arr))
        else:
            # column missing -> empty arrays for each row (keeps index correspondence)
            for idx in df.index:
                items.append((idx, np.array([], dtype=float)))
        mode_lists[i] = items

    return {"mode_series": mode_series, "mode_lists": mode_lists}

def condense_by_min_abs(data):
    """
    Condense each tuple (idx_group, combo, array) into
    (idx_group, combo, value) where value is the element with the smallest
    absolute magnitude, but preserve its original sign.
    """
    condensed = []
    for idx_group, combo, arr in data:
        # choose element with smallest abs() but keep real sign
        min_val = min(arr, key=lambda x: abs(x))
        condensed.append((idx_group, combo, min_val))
    return condensed

def filter_by_max_min_abs(data):
    """
    Filter condensed tuples so that only those whose stored value has the
    largest absolute magnitude remain. Original sign preserved.
    """
    if not data:
        return []
    
    max_abs = max(abs(t[2]) for t in data)
    return [t for t in data if abs(t[2]) == max_abs]




def combine_lists(*lists):
    """
    Fully general N-dimensional version that only allows
    element index combinations with unique indices.
    """

    # Number of lists (N)
    N = len(lists)

    # Length of the vectors (K)
    K = len(lists[0][0][1])

    # Collect all distinct original indices
    keys = [idx for idx, _ in lists[0]]

    # Map each list by idx for fast lookup
    idx_maps = []
    for lst in lists:
        idx_maps.append({idx: arr for idx, arr in lst})

    # Storage for output groups
    groups = {key: [] for key in keys}

    # Loop over each index group
    for key in keys:

        # Grab the vector chosen from each list for this group
        chosen = [idx_maps[m][key] for m in range(N)]

        # Sweep all element-index choices independently
        for elem_choices in product(range(K), repeat=N):

            # NEW RULE: require all unique indices
            if len(set(elem_choices)) != N:
                continue

            # Build output vector element-wise
            values = np.array([
                chosen[m][elem_choices[m]]
                for m in range(N)
            ])

            # Store tuple
            groups[key].append(
                (key, elem_choices, values)
            )

    return groups

def select_global_max_min_abs(groups, tol=1e-12):
    """
    Given a list of lists where each inner list contains tuples
    (idx_group, combo, value),
    return all tuples whose |value| equals the maximum absolute value
    across the entire dataset, allowing for floating-point tolerance.
    """

    # Flatten everything into one list of tuples
    all_tuples = [t for group in groups for t in group]

    if not all_tuples:
        return []

    # Compute global maximum |value|
    global_max = max(abs(t[2]) for t in all_tuples)

    # Collect all tuples that match this max within tolerance
    winners = [
        t for t in all_tuples
        if abs(abs(t[2]) - global_max) < tol
    ]

    return winners


In [4]:
N = np.arange(3,4)
P_opt = [100e-3]
w0 = 1e-6
f_rf_r = 1e6
N3= tweezer_combos_full_radial(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N,
    f_rf_r,
    P_opt,
    w0,
    max_tweezed=1,
)
N3

,N,Tweezed ions,P_per_tweezer (W),Combined radial frequencies,Mode0_freq,Mode0_eigvec,Mode1_freq,Mode1_eigvec,Mode2_freq,Mode2_eigvec
0,3,(),0.0,"[6283185.307179586, 6283185.307179586, 6283185...",1.000000e+06,"[0.577350269189621, 0.5773502691896278, 0.5773...",987253.616904,"[0.7071067811865754, -6.038822267752199e-14, -...",969127.076195,"[-0.4082482904638137, 0.8164965809277251, -0.4..."
1,3,"(0,)",0.1,"[6595799.989691874, 6283185.307179586, 6283185...",1.040601e+06,"[-0.9836895964499031, -0.17081996742552139, -0...",994305.089283,"[-0.135804001686669, 0.4998713272350667, 0.855...",971785.786802,"[0.11794935761636946, -0.849087271684716, 0.51..."
2,3,"(1,)",0.1,"[6283185.307179586, 6595799.989691874, 6283185...",1.034651e+06,"[0.21289466185572722, 0.9535993529290289, 0.21...",987253.616904,"[-0.7071067811869692, 1.7583265342019252e-13, ...",985234.808714,"[0.6742965689907784, -0.301078518153219, 0.674..."
3,3,"(2,)",0.1,"[6283185.307179586, 6283185.307179586, 6595799...",1.040601e+06,"[0.056349947337785096, 0.17081996742554714, 0....",994305.089283,"[-0.8553864210602142, -0.49987132723500277, 0....",971785.786802,"[0.5149162593090536, -0.8490872716847487, 0.11..."


In [13]:
result = build_mode_series_and_combinations(N3)
mode0_list = result["mode_lists"][0]
mode1_list = result["mode_lists"][1]
mode2_list = result["mode_lists"][2]
# inspect first row's Mode0 eigenvector and its original index
print(mode0_list)
print(mode1_list)
print(mode2_list)

[(0, array([0.57735027, 0.57735027, 0.57735027])), (1, array([-0.9836896 , -0.17081997, -0.05634995])), (2, array([0.21289466, 0.95359935, 0.21289466])), (3, array([0.05634995, 0.17081997, 0.9836896 ]))]
[(0, array([ 7.07106781e-01, -6.03882227e-14, -7.07106781e-01])), (1, array([-0.135804  ,  0.49987133,  0.85538642])), (2, array([-7.07106781e-01,  1.75832653e-13,  7.07106781e-01])), (3, array([-0.85538642, -0.49987133,  0.135804  ]))]
[(0, array([-0.40824829,  0.81649658, -0.40824829])), (1, array([ 0.11794936, -0.84908727,  0.51491626])), (2, array([ 0.67429657, -0.30107852,  0.67429657])), (3, array([ 0.51491626, -0.84908727,  0.11794936]))]


In [21]:
untweezed = combine_lists(mode0_list, mode1_list, mode2_list)[0]
tweeze0 = combine_lists(mode0_list, mode1_list, mode2_list)[1]
tweeze1 = combine_lists(mode0_list, mode1_list, mode2_list)[2]
tweeze2 = combine_lists(mode0_list, mode1_list, mode2_list)[3]
print(untweezed)

[(0, (0, 1, 2), array([ 5.77350269e-01, -6.03882227e-14, -4.08248290e-01])), (0, (0, 2, 1), array([ 0.57735027, -0.70710678,  0.81649658])), (0, (1, 0, 2), array([ 0.57735027,  0.70710678, -0.40824829])), (0, (1, 2, 0), array([ 0.57735027, -0.70710678, -0.40824829])), (0, (2, 0, 1), array([0.57735027, 0.70710678, 0.81649658])), (0, (2, 1, 0), array([ 5.77350269e-01, -6.03882227e-14, -4.08248290e-01]))]


In [23]:
min_untweezed = condense_by_min_abs(untweezed)
min_tweeze_0 = condense_by_min_abs(tweeze0)
min_tweeze_1 = condense_by_min_abs(tweeze1)
min_tweeze_2 = condense_by_min_abs(tweeze2)


In [27]:
best_untweezed = filter_by_max_min_abs(min_untweezed)
best_tweeze_0 = filter_by_max_min_abs(min_tweeze_0)
best_tweeze_1 = filter_by_max_min_abs(min_tweeze_1)
best_tweeze_2 = filter_by_max_min_abs(min_tweeze_2)
print(best_untweezed)
print(best_tweeze_0)
print(best_tweeze_1)
print(best_tweeze_2)

[(0, (2, 0, 1), 0.5773502691896285)]
[(1, (0, 2, 1), -0.849087271684716)]
[(2, (1, 0, 2), 0.6742965689916613)]
[(3, (2, 0, 1), -0.8490872716847487)]


In [31]:
select_global_max_min_abs([best_untweezed, best_tweeze_0, best_tweeze_1, best_tweeze_2])

[(1, (0, 2, 1), -0.849087271684716), (3, (2, 0, 1), -0.8490872716847487)]